In [0]:
%pip install s3fs openpyxl
%pip install openpyxl

In [0]:
import re
from io import BytesIO
import pandas as pd

ruta = "s3://amankay-datalake-2026/gold/integracion_de_datos/2026-10-23/integracion_2025_2026-10-23.xlsx"
contenido = spark.read.format("binaryFile").load(ruta).collect()[0]["content"]
pdf = pd.read_excel(BytesIO(contenido), engine="openpyxl")

pdf.columns = [re.sub(r"_+", "_", re.sub(r"[^0-9a-zA-Z]", "_", str(c).strip().lower())).strip("_") for c in pdf.columns]

for c in pdf.columns:
    if any(k in c for k in ["ubigeo", "codigo", "fecha", "departamento", "provincia", "distrito"]):
        pdf[c] = pdf[c].astype("string")
        continue
    num = pd.to_numeric(pdf[c], errors="coerce")
    if pdf[c].notna().any() and num.notna().sum() >= 0.9 * pdf[c].notna().sum():
        pdf[c] = num

df = spark.createDataFrame(pdf.astype(object).where(pdf.notna(), None))
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("big_data_team1.anghelo.integracion_gold")

print(len(pdf), "filas")
display(spark.table("big_data_team1.anghelo.integracion_gold").limit(10))